In [55]:
!pip install transformers datasets sentencepiece
!pip install nltk

In [56]:
from datasets import load_dataset

# Load the CNN/DailyMail dataset
dataset = load_dataset("cnn_dailymail", "3.0.0")

In [57]:
# Optional: Use a smaller subset for testing purposes
small_train = dataset["train"].shuffle(seed=42).select(range(500))  
small_val = dataset["validation"].shuffle(seed=42).select(range(100))  

In [58]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Choose the model and tokenizer
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [59]:
def preprocess_data(batch):
    inputs = [article for article in batch["article"]]
    targets = [highlights for highlights in batch["highlights"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")  # Reduced max_length

    # Tokenize targets as labels
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [60]:
# Tokenize the train and validation sets with optimizations
tokenized_train = small_train.map(
    preprocess_data,
    batched=True,
    batch_size=100,  # Adjust batch size based on memory capacity
    remove_columns=["article", "highlights", "id"],
    num_proc=4      # Number of CPU cores to use for parallel processing; adjust based on available cores
)

tokenized_val = small_val.map(
    preprocess_data,
    batched=True,
    batch_size=100,  # Adjust batch size based on memory capacity
    remove_columns=["article", "highlights", "id"],
    num_proc=4      # Number of CPU cores to use for parallel processing
)

Map (num_proc=4):   0%|          | 0/500 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

In [61]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    gradient_accumulation_steps=8,
    predict_with_generate=True,
    fp16=True  # Aktivera mixed precision för att minska minnesanvändning
)

In [62]:
!pip install evaluate
!pip install rouge_score absl-py

In [63]:
import evaluate

# Load the ROUGE metric
metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE scores
    result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {key: value.mid.fmeasure * 100 for key, value in result.items()}  # Scale to percentage

    return result

In [64]:
from transformers import Seq2SeqTrainer

# Initialize the Seq2SeqTrainer without the tokenizer argument
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)

In [65]:
trainer.train()

Epoch,Training Loss,Validation Loss


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

AttributeError: 'numpy.float64' object has no attribute 'mid'

In [73]:
import torch

# Set the index to select a specific sample
index = 2  # Adjust index to test different samples

# Select a sample article from the dataset
dialogue = small_train['article'][index][:1000]  # Truncate to 1000 characters if needed

# Prepare the prompt
prompt = f"""
Provide a concise summary for the following news article.

### Input:
{dialogue}

### Summary:
"""

# Tokenize the prompt and generate the output
input_ids = tokenizer(prompt, return_tensors='pt', truncation=True).input_ids

# Generate summary using the trained model
with torch.no_grad():
    outputs = model.generate(input_ids=input_ids, max_new_tokens=350)  # Adjust max_new_tokens if needed

# Decode and process the generated output
output = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Print the input prompt and the generated summary
dash_line = '-' * 100
print(dash_line)
print(f'INPUT PROMPT:\n{prompt}')
print(dash_line)
print(f'TRAINED MODEL GENERATED TEXT :\n{output}')

----------------------------------------------------------------------------------------------------
INPUT PROMPT:

Provide a concise summary for the following news article.

### Input:
Cover-up: Former Archbishop Lord Hope allowed a paedophile priest to escape punishment for sex crimes, a judge's report claims . A former archbishop who failed to act on alleged crimes of a paedophile priest should be jailed, the abuser’s victims have said. Lord Hope, the former Archbishop of York, did not act on 19 occasions when allegations of abuse or inappropriate conduct by the priest were raised with him, a scathing report revealed yesterday. But despite that, Lord Hope remains as an honorary assistant bishop in the Diocese of Bradford. Eli Ward and Bim Atkinson, who were among seven victims of the Very Reverend Robert Waddington, former Dean of Manchester, said it is ‘incredible’ that the man who did nothing to stop the priest has been allowed to keep his post. The report by Judge Sally Cahill QC

In [74]:
model.save_pretrained('results/bart_finetune_cnn')
tokenizer.save_pretrained('results/bart_finetune_cnn')

/Users/klarabratteby/Desktop/Skola/TNM114/LLM-Predictive-Text-Generator/gpt2_env/lib/python3.10/site-packages/transformers/modeling_utils.py:2817: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


('results/bart_finetune_cnn/tokenizer_config.json',
 'results/bart_finetune_cnn/special_tokens_map.json',
 'results/bart_finetune_cnn/vocab.json',
 'results/bart_finetune_cnn/merges.txt',
 'results/bart_finetune_cnn/added_tokens.json',
 'results/bart_finetune_cnn/tokenizer.json')